# HyperStreamDB: Full Pipeline Presentation

**Version:** 0.3.0 (Alpha)  
**Project:** HyperStreamDB - Serverless Index-Streaming Data Lake

This masterclass takes you from zero to a production-grade RAG pipeline, demonstrating hardware acceleration, hybrid search, and serverless table management.

In [1]:
import logging

# Silence Python-level logging
logging.basicConfig(level=logging.WARNING)
logging.getLogger('hyperstreamdb').setLevel(logging.WARNING)

print("HyperStreamDB Logging Level Python set to WARNING")

HyperStreamDB Logging Level Python set to WARNING


In [2]:
import logging
import hyperstreamdb as hdb

# Silence debug noise, only show WARNING and ERROR
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger('hyperstreamdb')
logger.setLevel(logging.WARNING)
print("HyperStreamDB Logging Level set to WARNING")

HyperStreamDB Logging Level set to WARNING


## 1. Setup & Environment

We initialize our environment and ensure all vector search dependencies are loaded.

In [3]:
import hyperstreamdb as hdb
import pyarrow as pa
import pandas as pd
import numpy as np
import os, shutil, requests, time, warnings
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 150)
# Silence Rust-level debug noise (crated by tracing in the core)
hdb.init_logging("WARN")

print("Environment ready.")

Environment ready.


In [4]:
hdb.__version__

'0.3.0'

## 2. Hardware Acceleration

HyperStreamDB can utilize Intel GPUs for massively parallel index construction. We check for available hardware here.

GPU support for Nvidia, Intel, Apple MPS, AMD(ROCm - untested)

In [5]:
import torch
device = "cpu"
if hdb.Device.is_available("cuda") or (hasattr(torch, "cuda") and torch.cuda.is_available()):
    device = "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")
ctx = hdb.Device(device)


Using device: cpu


## 3. Data Preparation (AG News)

We load the AG News dataset and generate semantic embeddings using a 384-dimensional SBERT model.

In [6]:
print("Loading dataset...")
dataset = load_dataset("ag_news", split="test[:500]")
df = pd.DataFrame(dataset)
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
df["category"] = df["label"].map(label_map)

print("Generating embeddings...")
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["text"].tolist())
df["embedding"] = [list(e) for e in embeddings]
print(f"Prepared {len(df)} records with {len(df['embedding'][0])}D vectors.")

df["id"] = range(len(df))


Loading dataset...
Generating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prepared 500 records with 384D vectors.


## 4. Table Creation (Serverless Iceberg)

HyperStreamDB tables are serverless and use standard Iceberg/Parquet formats with sidecar indices.

In [7]:
TABLE_URI = "file:///tmp/hdb_masterclass"
if os.path.exists("/tmp/hdb_masterclass"): shutil.rmtree("/tmp/hdb_masterclass")

schema = pa.schema([
    ('id', pa.int32()), 
    ('category', pa.string()),
    ('text', pa.string()),
    ('embedding', pa.list_(pa.float32(), 384))
])

table = hdb.Table.create(TABLE_URI, schema, device=ctx)
print(f"Table created at {TABLE_URI}")

Table created at file:///tmp/hdb_masterclass


## 5. Ingestion & Indexing

We write the data and building HNSW vector indices and Inverted scalar indices simultaneously.

In [8]:
table.add_index_columns(["embedding", "category", "text"])
table.write(df[["id", "category", "text", "embedding"]])
table.commit()
print("Data ingested and indexed successfully.")

Data ingested and indexed successfully.


## 6. Hybrid Search Performance

We perform a semantic search combined with a scalar filter to find 'Space' news in the 'Sci/Tech' category.

In [9]:
query_vec = list(model.encode("Latest milestones in space exploration"))

results = (table.filter("category = 'Sci/Tech'")
                 .vector_search(query_vec, k=3)
                 .to_pandas())

print("Hybrid Search Results (Sci/Tech + Vector):")
display(results[["category", "text", "distance"]])

Smart Trigger Check: Column 'category' has BM25 index: false
  Found indexes: []
2026-04-15T13:12:33.995681Z ERROR hyperstreamdb::core::reader: Vector index listed in manifest failed, falling back to flat scan: Object at location /tmp/hdb_masterclass/seg_0e9c959b-2249-4125-9312-8f575da1cb4d.embedding.tq8 not found: No such file or directory (os error 2)
Hybrid Search Results (Sci/Tech + Vector):


,category,text,distance
0,Sci/Tech,"The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers ...",1.113737
1,Sci/Tech,Stunt pilots to snag a falling NASA craft NASA #39;s three-year effort to bring some genuine star dust back to Earth is set for a dramatic finale ...,1.254462
2,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space Agency -- ESAs Mars Express has relayed pictures from one of NASA's Mars rovers for t...,1.255060


## 7. SQL & pgvector Interface

HyperStreamDB supports standard SQL with pgvector-compatible operators for vector distance.

In [10]:
session = hdb.Session()
session.register("news", table)

query_vec_str = str([float(x) for x in query_vec]).replace(' ', '')
sql_df = session.sql(f"""
    SELECT category, text, 
           embedding <-> '{query_vec_str}'::vector AS dist
    FROM news
    WHERE category = 'Sci/Tech'
    ORDER BY dist ASC LIMIT 3
""")
display(sql_df.to_pandas())

,category,text,dist
0,Sci/Tech,"The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers ...",1.055338
1,Sci/Tech,Stunt pilots to snag a falling NASA craft NASA #39;s three-year effort to bring some genuine star dust back to Earth is set for a dramatic finale ...,1.120027
2,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space Agency -- ESAs Mars Express has relayed pictures from one of NASA's Mars rovers for t...,1.120294


## 8. Vector Aggregations

Calculate semantic centroids for entire categories directly in SQL.

In [11]:
agg_df = session.sql("""
    SELECT category, 
           vector_avg(embedding) AS centroid,
           COUNT(*) as count
    FROM news
    GROUP BY category
    order by count asc
""").to_pandas()
display(agg_df[["category", "count", "centroid"]])

,category,count,centroid
0,Business,106,"[-0.009712125, -0.019897953, 0.021966765, 0.01247102, 0.014252397, 0.00058017054, -0.008388984, 0.0053979103, -0.0032875366, -0.00044038688, 0.006..."
1,World,122,"[-4.846262e-05, 0.020821093, 0.009778806, 0.0045922627, 0.018377043, -0.0015206813, 0.01483018, -0.029480884, -0.0034735817, 0.026525706, 0.002213..."
2,Sci/Tech,127,"[-0.022988431, -0.006676947, 0.017107774, -0.013084924, 0.018412959, -0.005346965, 0.0069102217, 0.004688753, -0.0051615434, 0.010351755, 0.012357..."
3,Sports,145,"[-0.0013631942, 0.044090025, 0.0071726996, -0.016102802, 0.00725294, 0.019899301, 0.04513628, 0.0041184537, 0.002906757, 0.009585048, -0.039248083..."


## 9. Table Maintenance

Keep the data lake healthy by compacting small files and vacuuming history.

In [13]:
print(f"Files before: {len(table.manifest()['entries'])}")
table.compact()
table.commit()
print(f"Files after: {len(table.manifest()['entries'])}")

table.vacuum(retention_versions=1)
print("Vacuum complete.")

Files before: 1
Files after: 1
Vacuum complete.


## 10. Advanced Partitioning

Scale to massive datasets using identity partitioning on the category field.

In [16]:
PART_URI = "file:///tmp/part_demo"
if os.path.exists("/tmp/part_demo"): shutil.rmtree("/tmp/part_demo")

partition_spec = {
    "fields": [{"name": "category", "transform": "identity", "source_id": 2, "field_id": 1000}]
}

part_table = hdb.Table.create_partitioned(PART_URI, schema, partition_spec)
part_table.write(df[:100])
part_table.commit()

print("Partitioned physical layout:")
for entry in part_table.manifest()['entries']:
    print(f" - {entry['file_path'].split('/')[-2:]}")

  [K-Means] Converged early at iteration 2
  [K-Means] Converged early at iteration 2
  [K-Means] Converged early at iteration 2
  [K-Means] Converged early at iteration 2
Partitioned physical layout:
 - ['category=World', 'seg_7e20ab97-6a2b-4cfa-a07f-56109e904ae5.parquet']
 - ['category=Business', 'seg_fc6cdbd6-4f00-419a-ac46-0f437be4e6ac.parquet']
 - ['Tech', 'seg_b92fde8b-29bf-4459-a543-56df561d9742.parquet']
 - ['category=Sports', 'seg_4a9f742d-1d8f-4292-bfbe-50186589412e.parquet']


## 11. End-to-End RAG Pipeline

Using SQuAD validation data to build a knowledge-backed Q&A system.

In [17]:
print("Loading SQuAD contexts...")
squad = load_dataset("squad", split="validation[0:100]")
squad_df = pd.DataFrame(squad).drop_duplicates(subset=["context"])

squad_emb = model.encode(squad_df["context"].tolist(), show_progress_bar=True)
squad_df["embedding"] = [list(e) for e in squad_emb]

rag_table = hdb.Table("rag_db_presentation")
rag_table.add_index_columns(["embedding", "context"])
rag_table.write(squad_df[["context", "title", "embedding"]])
rag_table.commit()
print("RAG knowledge base initialized.")

Loading SQuAD contexts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  [K-Means] Converged early at iteration 2
  [K-Means] Converged early at iteration 2
  [K-Means] Converged early at iteration 2
  [K-Means] Converged early at iteration 2
RAG knowledge base initialized.


## 12. Inference with Retrieval

Connect retrieval to the LLM (requires GROQ_API_KEY).

In [18]:
def ask_hdb(question):
    # Convert question to an embedding (using your local model)
    q_emb = model.encode(question)
    
    # Perform a vector search on your table
    # This is where your table data gets pulled in!
    ctx_df = rag_table.to_pandas(vector_filter={
        "column": "embedding", 
        "query": q_emb, 
        "k": 3  # Get top 3 matching contexts
    })
    
    # Combine retrieved contexts into one string
    contexts = ctx_df["context"].tolist()
    prompt = f"Answer the question based ONLY on this context:\n\n" + "\n".join(contexts) + f"\n\nQuestion: {question}"
    
    # Use a supported Groq model
    model_name = "llama-3.3-70b-versatile"
    
    api_key = os.environ.get("GROQ_TOKEN")
    r = requests.post("https://api.groq.com/openai/v1/chat/completions", 
                      headers={"Authorization": f"Bearer {api_key}"},
                      json={"model": model_name, "messages": [{"role": "user", "content": prompt}]})
    
    res = r.json()
    if "choices" not in res:
        # This will show you the exact error from Groq (e.g., "model_not_found")
        print(f"Groq API Error: {res}")
        return "Error from API"
        
    return res["choices"][0]["message"]["content"]

In [19]:
question = "Where were the first modern Olympic games?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

2026-04-15T13:16:32.578662Z ERROR hyperstreamdb::core::reader: Vector index listed in manifest failed, falling back to flat scan: Object at location /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_a0febfe0-23da-4395-a46d-e14bc1376183.embedding.tq8 not found: No such file or directory (os error 2)
2026-04-15T13:16:32.578764Z ERROR hyperstreamdb::core::reader: Vector index listed in manifest failed, falling back to flat scan: Object at location /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_ea71a295-e0ce-4723-a4ab-41717dc8d4fb.embedding.tq8 not found: No such file or directory (os error 2)
2026-04-15T13:16:32.578783Z ERROR hyperstreamdb::core::reader: Vector index listed in manifest failed, falling back to flat scan: Object at location /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_117667b0-8d96-4c63-b263-0349c10fe4e8.embedding.tq8 not found: No such file or directory (os error 2)
2026-04-15T13:16:32.578803Z ERROR hyperstreamd

In [20]:
question = "What city was superbowl 50 played at?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

2026-04-15T13:16:33.874004Z ERROR hyperstreamdb::core::reader: Vector index listed in manifest failed, falling back to flat scan: Object at location /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_a0febfe0-23da-4395-a46d-e14bc1376183.embedding.tq8 not found: No such file or directory (os error 2)
2026-04-15T13:16:33.874130Z ERROR hyperstreamdb::core::reader: Vector index listed in manifest failed, falling back to flat scan: Object at location /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_117667b0-8d96-4c63-b263-0349c10fe4e8.embedding.tq8 not found: No such file or directory (os error 2)
2026-04-15T13:16:33.874177Z ERROR hyperstreamdb::core::reader: Vector index listed in manifest failed, falling back to flat scan: Object at location /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_4064100a-da86-4526-a78b-0e0c7d5ea49d.embedding.tq8 not found: No such file or directory (os error 2)
2026-04-15T13:16:33.874214Z ERROR hyperstreamd